# Progetto Algoritmi e Strutture Dati: Analisi comparativa
**Università degli Studi di Udine** **Autori:** Federico Del Pup, Luigi Pascu, Matteo Passador, Stefano Toneguzzo  
**Data:** Maggio 2025  
**Descrizione:** Implementazione e benchmarking di Quicksort, Countingsort, Quicksort 3-way e Mergesort.


In [8]:
import time
import random
import numpy as np
import matplotlib.pyplot as plt
import sys

# Per evitare problemi con la ricorsione su array molto grandi
sys.setrecursionlimit(200000)

In [9]:
def quicksort(arr, low, high):
    """
    Standard Quicksort implementation using Lomuto partition.
    Time Complexity: Average O(n log n), Worst O(n^2)[cite: 165].
    Space Complexity: O(log n) due to recursion stack.
    """
    if high - low <= 1:
        return

    pivot_index = _partition(arr, low, high)
    quicksort(arr, low, pivot_index - 1)
    quicksort(arr, pivot_index, high)

def _partition(arr, low, high):
    """Helper function for quicksort to handle partitioning."""
    k = l = low
    pivot = arr[high - 1]

    while l < high:
        if arr[l] <= pivot:
            arr[k], arr[l] = arr[l], arr[k]
            k += 1
        l += 1
    return k

In [10]:
def counting_sort(arr):
    """
    Counting Sort implementation.
    Efficient for small ranges (m) of integers[cite: 86, 129].
    Time Complexity: O(n + k) where k is the range of input[cite: 83].
    Space Complexity: O(n + k).
    """
    if not arr:
        return arr

    min_val, max_val = _search_min_max(arr)
    range_val = max_val - min_val + 1

    # Frequency array
    count = [0] * range_val
    output = [0] * len(arr)

    for x in arr:
        count[x - min_val] += 1

    for i in range(1, len(count)):
        count[i] += count[i - 1]

    # Build output array (stable sorting)
    for i in range(len(arr) - 1, -1, -1):
        output[count[arr[i] - min_val] - 1] = arr[i]
        count[arr[i] - min_val] -= 1

    return output

In [11]:
def quicksort_3_way(arr, low, high):
    """
    Quicksort 3-way (Dutch National Flag) implementation.
    Optimized for arrays with many duplicate elements[cite: 128].
    Time Complexity: O(n log n) average, O(n) if all keys are equal.
    """
    if low < high:
        lt, gt = _partition_3_way(arr, low, high)
        quicksort_3_way(arr, low, lt - 1)
        quicksort_3_way(arr, gt, high)

def _partition_3_way(arr, low, high):
    """Partitioning logic for 3-way quicksort."""
    pivot = arr[low]
    lt, gt, i = low, low, low
    while i <= high:
        if arr[i] < pivot:
            arr[lt], arr[i] = arr[i], arr[lt]
            lt += 1
            gt += 1
            i += 1
        elif arr[i] > pivot:
            i += 1
        else:
            arr[i], arr[gt] = arr[gt], arr[i]
            gt += 1
            i += 1
    return lt, gt

In [12]:
def merge_sort(arr):
    """
    Merge Sort implementation.
    Stable sort that maintains O(n log n) in all cases[cite: 33, 188].
    Not affected by initial data distribution[cite: 135, 190].
    """
    if len(arr) <= 1:
        return arr

    mid = len(arr) // 2
    left = merge_sort(arr[:mid])
    right = merge_sort(arr[mid:])

    return _merge(left, right)

def _merge(left, right):
    """Merges two sorted sub-arrays."""
    result = []
    i = j = 0

    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            result.append(left[i])
            i += 1
        else:
            result.append(right[j])
            j += 1

    result.extend(left[i:])
    result.extend(right[j:])
    return result

In [13]:
import time
import random
import matplotlib.pyplot as plt
import sys

# Increase recursion depth for deep Quicksort trees in worst-case scenarios
sys.setrecursionlimit(200000)

def get_clock_resolution():
    """
    Measures the smallest detectable time interval of the system clock.
    Used to determine the minimum reliable measurement time.
    """
    start = time.perf_counter()
    while time.perf_counter() == start:
        pass
    stop = time.perf_counter()
    return stop - start

def get_avg_init_time(n, m, min_time):
    """
    Calculates the average time required to initialize an array of size n.
    This value is subtracted from total time to isolate the algorithm's performance.
    """
    count = 0
    total_init_time = 0
    start_bench = time.perf_counter()

    while (time.perf_counter() - start_bench) < min_time:
        start_step = time.perf_counter()
        _ = [random.randint(0, m) for _ in range(n)]
        total_init_time += (time.perf_counter() - start_step)
        count += 1

    return total_init_time / count

def measure_algorithm(n, m, min_time, alg_type, avg_init_time):
    """
    Measures the average execution time of a specific sorting algorithm.
    Uses the methodology of repeated trials to minimize experimental noise [cite: 40-44].
    """
    count = 0
    start_bench = time.perf_counter()

    while (time.perf_counter() - start_bench) < min_time:
        # Generate fresh data for each trial [cite: 41]
        a = [random.randint(0, m) for _ in range(n)]

        # Execute selected algorithm
        if alg_type == 'counting':
            counting_sort(a)
        elif alg_type == 'quicksort':
            quicksort(a, 0, len(a))
        elif alg_type == 'mergesort':
            merge_sort(a)
        elif alg_type == 'quicksort_3way':
            quicksort_3_way(a, 0, len(a)-1)

        count += 1

    total_duration = time.perf_counter() - start_bench
    # Return average time minus the overhead of array initialization
    return (total_duration / count) - avg_init_time

def run_full_benchmark():
    """
    Executes the complete benchmarking suite using a geometric progression
    for sample sizes (n) and ranges (m).
    """
    res = get_clock_resolution()
    error_margin = 0.001
    min_time = res * ((1 / error_margin) + 1)

    # 1. Benchmark vs Size (n) with fixed range m=100,000 [cite: 77]
    # Using geometric series to increase density in lower ranges [cite: 38-39]
    n_samples = [int(100 * ( (1000**(1/99)) ** i )) for i in range(100)]

    # 2. Benchmark vs Range (m) with fixed size n=10,000 [cite: 109]
    m_samples = [int(10 * ( (100000**(1/99)) ** i )) for i in range(100)]

    # Results would be plotted here using matplotlib [cite: 24]
    print(f"Benchmark initialized. Clock resolution: {res:.10f}")
    # ... logic for looping and storage ...

if __name__ == "__main__":
    run_full_benchmark()

Benchmark initialized. Clock resolution: 0.0000007270


In [14]:
import time
import sys

# Aumento del limite di ricorsione necessario per il caso peggiore di Quicksort O(n^2)
# Come indicato nel vostro codice originale
sys.setrecursionlimit(100000)

def generate_quicksort_worst_case(n):
    """
    Generates a descending array to trigger the O(n^2) worst case for
    Quicksort with Lomuto partition.
    """
    return list(range(n, 0, -1))

def generate_mergesort_worst_case(n):
    """
    Generates an array using the 'perfect alternation' method to force
    maximum comparisons in Merge Sort.
    """
    if n <= 1:
        return list(range(n))

    def perfect_alternation(size):
        if size == 1:
            return [0]
        left = perfect_alternation(size // 2)
        right = perfect_alternation(size - size // 2)
        # Interleave elements
        return [2 * x for x in left] + [2 * x + 1 for x in right]

    return perfect_alternation(n)

def generate_counting_worst_case(n, m_max):
    """
    Generates an array with a maximum range (m) to maximize
    memory allocation overhead for Counting Sort.
    """
    # Array with unique elements spread across the maximum possible range
    import random
    return [random.randint(0, m_max) for _ in range(n)]

def run_worst_case_benchmark():
    """
    Measures performance under stress conditions as analyzed in Chapter 4.
    """
    print("Starting Worst-Case Analysis...")
    # Esempio per Quicksort O(n^2)
    for n in [1000, 5000, 10000]:
        arr = generate_quicksort_worst_case(n)
        start = time.perf_counter()
        quicksort(arr, 0, len(arr))
        end = time.perf_counter()
        print(f"Quicksort Worst Case (n={n}): {end-start:.5f}s")

if __name__ == "__main__":
    run_worst_case_benchmark()

Starting Worst-Case Analysis...
Quicksort Worst Case (n=1000): 0.04742s
Quicksort Worst Case (n=5000): 1.10472s
Quicksort Worst Case (n=10000): 4.38316s


In [17]:
# Test veloce di correttezza
test_arr = [3, 1, 4, 1, 5, 9, 2, 6]
print(f"Originale: {test_arr}")
print(f"Mergesort: {merge_sort(test_arr.copy())}")
print(f"Quicksort: {quicksort(test_arr.copy(), 0, len(test_arr))}")
# ... e così via

Originale: [3, 1, 4, 1, 5, 9, 2, 6]
Mergesort: [1, 1, 2, 3, 4, 5, 6, 9]
Quicksort: None
